# Day 1 — Python: For Loop, Break, Continue, Enumerate, Zip
### Target: Data Engineer Interviews

> **Roadmap Day:** 1 · **Date:** Monday, June 15, 2026  
> **Prerequisite:** Run `00_prerequisites.ipynb` first (installs packages, creates sample data files).

Run each cell in order. Try writing your solution to each problem before looking at the solution cell.

---
## 1. Basic `for` Loop

Python `for` is always **for-each** — iterates over any iterable.  
No index counter. No length check. No off-by-one errors.

In [ ]:
# Iterating a list
products = ["laptop", "mouse", "keyboard", "monitor"]

for product in products:
    print(product)

In [ ]:
# Iterating a string — character by character
for char in "DATA":
    print(char, end=" ")
print()   # newline

In [ ]:
# Iterating range — most common for numbered steps
for i in range(1, 6):       # 1, 2, 3, 4, 5 — 6 is excluded
    print(i, end=" ")
print()

In [ ]:
# Iterating dict — key/value pairs
db_config = {"host": "localhost", "port": 5432, "db": "orders"}

for key, value in db_config.items():
    print(f"  {key}: {value}")

---
## 2. `break` — Exit the Loop Early

`break` immediately exits the **innermost** loop.  
Remaining iterations are skipped entirely.

**DE context:** Stop reading chunks once you've collected enough records. Stop polling an API on first success.

In [ ]:
# Stop on first failure
pipeline_stages = ["ingest", "validate", "transform", "load"]
failed_stages   = {"transform"}

for stage in pipeline_stages:
    print(f"Running: {stage}")
    if stage in failed_stages:
        print(f"  >> Stage '{stage}' failed. Aborting pipeline.")
        break

In [ ]:
# Accumulate until threshold — DE pattern
daily_sales = [1200, 3400, 2100, 800, 4500, 1900, 2200]
running_total = 0

print(f"{'Day':<5} {'Sales':>8} {'Running Total':>15}")
print("-" * 30)

for day, sales in enumerate(daily_sales, start=1):
    running_total += sales
    print(f"Day {day:<2} {sales:>8,} {running_total:>15,}")
    if running_total > 10000:
        print(f"\n>> Crossed 10,000 on Day {day} — total = {running_total:,}")
        break

---
## 3. `continue` — Skip This Iteration

`continue` skips the rest of the current iteration's body and jumps to the next iteration.  
The loop does NOT exit.

**DE context:** Skip null/bad rows without breaking the pipeline.

In [ ]:
# Skip None values — typical in data cleaning
amounts = [500, None, 1200, None, 800, 300, None, 450]

total      = 0
null_count = 0

for amount in amounts:
    if amount is None:
        null_count += 1
        continue           # skip the processing — go to next iteration
    total += amount

print(f"Total (excluding nulls): {total:,}")
print(f"Null values skipped: {null_count}")

In [ ]:
# Skip invalid records and collect error log
VALID_DEPTS = {"Engineering", "Finance", "HR", "Marketing", "Operations"}

records = [
    {"id": 1, "name": "Alice",  "dept": "Engineering"},
    {"id": 2, "name": "Bob",    "dept": "Unknown"},
    {"id": 3, "name": "Carol",  "dept": "Finance"},
    {"id": 4, "name": "Dave",   "dept": None},
    {"id": 5, "name": "Eve",    "dept": "HR"},
    {"id": 6, "name": "Frank",  "dept": "INVALID"},
]

clean  = []
errors = []

for rec in records:
    if rec["dept"] not in VALID_DEPTS:
        errors.append({"id": rec["id"], "name": rec["name"], "bad_dept": rec["dept"]})
        continue
    clean.append(rec)

print(f"Clean records: {len(clean)}")
print(f"Errored records: {len(errors)}")
print(f"Errors: {errors}")

---
## 4. `for / else` — The Loop's Else Clause

The `else` block runs **only if the loop completed without hitting a `break`**.  
Useful for signaling "not found" without a flag variable.

In [ ]:
# Search for a record — for/else avoids a flag variable
employees = [
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"},
    {"id": 3, "name": "Carol"},
]

target_id = 99

for emp in employees:
    if emp["id"] == target_id:
        print(f"Found: {emp['name']}")
        break
else:
    print(f"Employee id={target_id} not found.")   # runs because no break was hit

In [ ]:
# Same result but with a found flag — more verbose
found = False
for emp in employees:
    if emp["id"] == target_id:
        print(f"Found: {emp['name']}")
        found = True
        break

if not found:
    print(f"Employee id={target_id} not found.")

---
## 5. `enumerate` — Loop With Index

`enumerate(iterable, start=0)` yields `(index, value)` pairs.  
Use whenever you need both the position and the value.

In [ ]:
products = ["laptop", "mouse", "keyboard", "monitor", "webcam"]

# Default start=0
print("0-based index:")
for idx, product in enumerate(products):
    print(f"  {idx}: {product}")

In [ ]:
# 1-based index — common in reports
print("Product Catalog")
print("-" * 20)
for idx, product in enumerate(products, start=1):
    print(f"[{idx:02d}] {product.title()}")

In [ ]:
# range(len()) vs enumerate — always prefer enumerate
print("range(len()) — avoid:")
for i in range(len(products)):
    print(f"  {i}: {products[i]}")

print("\nenumerate — prefer:")
for i, product in enumerate(products):
    print(f"  {i}: {product}")

In [ ]:
# DE use case — track which row number had a DQ error
rows = [
    {"order_id": 101, "amount": 500},
    {"order_id": 102, "amount": -50},
    {"order_id": 103, "amount": 1200},
    {"order_id": 104, "amount": 0},
    {"order_id": 105, "amount": 800},
]

dq_errors = []
for row_num, row in enumerate(rows, start=1):
    if row["amount"] <= 0:
        dq_errors.append({
            "row": row_num,
            "order_id": row["order_id"],
            "issue": f"amount={row['amount']} must be > 0"
        })

print(f"DQ report: {len(dq_errors)} error(s) found")
for e in dq_errors:
    print(f"  Row {e['row']} | order_id={e['order_id']} | {e['issue']}")

---
## 6. `zip` — Pair Two or More Iterables

`zip(a, b)` pairs elements by position.  
**Stops at the shortest iterable** — extra elements in the longer one are silently dropped.

In [ ]:
# Basic zip
names    = ["Alice", "Bob", "Carol", "David"]
salaries = [80000, 95000, 72000, 110000]

print(f"{'Employee':<12} {'Salary':>12}")
print("-" * 26)
for name, salary in zip(names, salaries):
    print(f"{name:<12} ${salary:>11,}")

In [ ]:
# zip truncates at the shortest — important to know
a = [1, 2, 3, 4, 5]
b = ["x", "y"]          # only 2 elements

result = list(zip(a, b))
print(f"zip result: {result}")          # only 2 pairs — 3, 4, 5 are dropped
print(f"len(a)={len(a)}, len(b)={len(b)}, len(zip)={len(result)}")

In [ ]:
# zip_longest — keep all elements with a fill value
from itertools import zip_longest

a = [1, 2, 3, 4, 5]
b = ["x", "y"]

result = list(zip_longest(a, b, fillvalue=None))
print(result)   # [(1,'x'), (2,'y'), (3,None), (4,None), (5,None)]

In [ ]:
# zip → dict — build structured records from headers + data rows
headers = ["emp_id", "name", "dept", "salary"]
values  = [101, "Alice Johnson", "Engineering", 95000]

record = dict(zip(headers, values))
print(record)

In [ ]:
# DE use case — read a CSV without pandas
# First row = headers, remaining rows = data
csv_rows = [
    ["emp_id", "name",          "dept",          "salary"],
    [1,         "Alice Johnson", "Engineering",   95000],
    [2,         "Bob Smith",     "Marketing",     72000],
    [3,         "Carol White",   "Finance",       85000],
]

headers  = csv_rows[0]
data_rows = csv_rows[1:]

records = [dict(zip(headers, row)) for row in data_rows]

for rec in records:
    print(rec)

In [ ]:
# Zip three iterables
names  = ["Alice", "Bob", "Carol"]
depts  = ["Engineering", "Marketing", "HR"]
sals   = [95000, 72000, 48000]

for name, dept, sal in zip(names, depts, sals):
    print(f"{name} | {dept} | ${sal:,}")

---
## 7. Unzipping — Reverse of `zip`

In [ ]:
# zip creates pairs, zip(*) undoes it
pairs = [("Alice", 80000), ("Bob", 95000), ("Carol", 72000)]

names, salaries = zip(*pairs)
print("Names:   ", names)
print("Salaries:", salaries)

In [ ]:
# Transpose a matrix using zip(*)
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

transposed = list(zip(*matrix))
print("Original matrix:")
for row in matrix:
    print(" ", row)

print("\nTransposed:")
for row in transposed:
    print(" ", row)

---
## 8. Combined Patterns — Real DE Usage

In [ ]:
# Pattern: read rows, skip nulls, stop after MAX valid records collected
raw_records = [
    {"id": 1, "amount": 500},
    {"id": 2, "amount": None},
    {"id": 3, "amount": 1200},
    {"id": 4, "amount": 800},
    {"id": 5, "amount": 300},
    {"id": 6, "amount": None},
    {"id": 7, "amount": 600},
]

MAX_GOOD = 3
good     = []
skipped  = 0

for rec in raw_records:
    if rec["amount"] is None:
        skipped += 1
        continue
    good.append(rec)
    if len(good) >= MAX_GOOD:
        break

print(f"Collected {len(good)} valid records, skipped {skipped} nulls")
for r in good:
    print(f"  id={r['id']}, amount={r['amount']:,}")

In [ ]:
# Pattern: compute running stats with enumerate
transactions = [120, 340, 80, 500, 220, 410, 90, 600]

running_total = 0
running_max   = float("-inf")

print(f"{'Tx#':<5} {'Amount':>8} {'Total':>10} {'Avg':>10} {'Max':>8}")
print("-" * 45)

for i, tx in enumerate(transactions, start=1):
    running_total += tx
    running_max    = max(running_max, tx)
    running_avg    = running_total / i
    print(f"{i:<5} {tx:>8,} {running_total:>10,} {running_avg:>10.1f} {running_max:>8,}")

In [ ]:
# Pattern: formatted report — enumerate + zip
names  = ["Alice Johnson", "Bob Smith", "Carol White", "Dave Brown"]
depts  = ["Engineering", "Marketing", "Finance", "HR"]
sals   = [95000, 72000, 85000, 60000]

print(f"{'#':<4} {'Name':<16} {'Department':<14} {'Salary':>10}")
print("-" * 48)

for i, (name, dept, sal) in enumerate(zip(names, depts, sals), start=1):
    print(f"{i:<4} {name:<16} {dept:<14} ${sal:>9,}")

---
## 9. Day 1 Problems — Official Solutions

Try writing your solution before running each cell.

In [ ]:
# Problem 1 (Easy)
# Using enumerate, loop over a list of products and print index + product name in a formatted way

products = ["laptop", "mouse", "keyboard", "monitor", "webcam", "headset"]

print("Product Catalog")
print("=" * 22)
for idx, product in enumerate(products, start=1):
    print(f"[{idx:02d}] {product.title()}")

In [ ]:
# Problem 2 (Easy)
# Using zip, pair employee names with their salaries and print a combined report

employees = ["Alice Johnson", "Bob Smith", "Carol White", "Dave Brown"]
salaries  = [95000, 65000, 85000, 60000]

print(f"{'Employee':<16} {'Salary':>12}")
print("-" * 30)

total = 0
for name, salary in zip(employees, salaries):
    print(f"{name:<16} ${salary:>11,}")
    total += salary

print("-" * 30)
print(f"{'Total payroll':<16} ${total:>11,}")
print(f"{'Average':<16} ${total // len(salaries):>11,}")

In [ ]:
# Problem 3 (Medium)
# Loop over daily sales; accumulate running total;
# break when total exceeds 10000 — report which day crossed the limit

daily_sales = [1200, 3400, 2100, 800, 4500, 1900, 2200]
running_total = 0

print(f"{'Day':<6} {'Sales':>8} {'Running Total':>15} {'Status':>10}")
print("-" * 42)

for day, sales in enumerate(daily_sales, start=1):
    running_total += sales
    crossed = running_total > 10000
    print(f"Day {day:<3} {sales:>8,} {running_total:>15,} {'<< CROSSED' if crossed else '':>10}")
    if crossed:
        print(f"\nResult: 10,000 crossed on Day {day}")
        print(f"  Sales that day : {sales:,}")
        print(f"  Running total  : {running_total:,}")
        print(f"  Overshoot by   : {running_total - 10000:,}")
        break
else:
    print("\n10,000 was never reached in this dataset.")

---
## 10. Interview Gotchas — Know These Cold

In [ ]:
# Gotcha 1: zip truncates at shortest — know this before using
a = [1, 2, 3, 4, 5]
b = ["x", "y"]

print(f"zip([1..5], ['x','y']) = {list(zip(a, b))}")
print(f"Only {len(list(zip(a, b)))} pairs — {len(a) - len(b)} elements from 'a' were dropped silently")

In [ ]:
# Gotcha 2: for/else — when does else run?
nums = [1, 2, 3]

print("Case A — loop finishes normally:")
for x in nums:
    pass
else:
    print("  else block RUNS (no break hit)")

print("\nCase B — loop breaks:")
for x in nums:
    if x == 2:
        break
else:
    print("  else block RUNS")   # won't print
print("  else block SKIPPED (break was hit)")

In [ ]:
# Gotcha 3: break only exits the innermost loop
# Use a flag or a function to break out of nested loops

matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
target = 5
found  = False

for row_i, row in enumerate(matrix):
    for col_j, val in enumerate(row):
        if val == target:
            found = True
            break         # exits inner loop only
    if found:
        break             # now exits outer loop

print(f"Found {target} at row={row_i}, col={col_j}")

In [ ]:
# Gotcha 4: enumerate start= defaults to 0, not 1
items = ["a", "b", "c"]

print("Default start=0:", list(enumerate(items)))
print("With start=1:   ", list(enumerate(items, start=1)))

In [ ]:
# Gotcha 5: unzip with zip(*) — interviewer may ask this
pairs = [("Alice", 80000), ("Bob", 95000), ("Carol", 72000)]

# zip(*pairs) is the unzip pattern
names, sals = zip(*pairs)
print(f"Unzipped names:    {names}")
print(f"Unzipped salaries: {sals}")

---
## Quick Cheat Sheet

```python
# Basic for loop
for item in iterable: ...

# Break — exit early
for item in iterable:
    if condition: break

# Continue — skip this iteration
for item in iterable:
    if bad_row: continue
    process(item)

# for/else — else runs only if no break hit
for item in iterable:
    if found: break
else:
    handle_not_found()

# Enumerate — with index
for i, val in enumerate(lst, start=1): ...

# Zip — pair iterables
for a, b in zip(list_a, list_b): ...

# Zip + enumerate
for i, (a, b) in enumerate(zip(la, lb), start=1): ...

# zip → dict
record = dict(zip(headers, values))

# Unzip
keys, vals = zip(*pairs)

# zip_longest — don't truncate
from itertools import zip_longest
for a, b in zip_longest(la, lb, fillvalue=None): ...
```